# `benchmark_baz.py`

按功能分段；每段前有标题，方便查找和修改。

### Imports & constants

In [1]:
# benchmark_baz.py
# 论文 benchmark: Long Only / Sgn(Returns) / MACD (Baz et al. 2015)
# 与神经网络用「完全相同的 (Date, Ticker) OOS 样本」，保证对比公平。

import numpy as np
import pandas as pd

from neural_config import FEATURES, TARGET_ANNUAL_VOL, TRADING_DAYS, MACD_PAIRS
from neural_data import load_model_data
from neural_backtest import (
    build_daily_portfolio, rescale_to_target_vol, summary_table,
)

TARGET_DAILY_VOL = TARGET_ANNUAL_VOL / np.sqrt(TRADING_DAYS)

### `phi()`

论文式 (7) 及 Exhibit 1: y*exp(-y^2/4)/0.89

In [2]:
def phi(y):
    """论文式 (7) 及 Exhibit 1: y*exp(-y^2/4)/0.89"""
    return y * np.exp(-(y ** 2) / 4) / 0.89

### `positions()`

In [3]:
def positions(df, rule):
    if rule == "long_only":
        return np.ones(len(df))
    if rule == "sgn_returns":                      # Moskowitz 2012, 过去一年收益符号
        return np.sign(df["norm_ret_1y"].to_numpy())
    if rule == "macd_baz":                         # Baz 2015, 式 (7)(8)
        y = sum(df[f"macd_{s}_{l}"] for s, l in MACD_PAIRS)
        return phi(y).to_numpy()
    raise ValueError(rule)

### `main()`

In [4]:
def main(ref_pred_path="lstm_oos_predictions.csv"):
    import os
    from neural_config import DATA_PATH
    df = load_model_data(DATA_PATH)

    if os.path.exists(ref_pred_path):
        # 已有神经网络预测：与其共用同一批 OOS 样本（最公平）
        ref = pd.read_csv(ref_pred_path, parse_dates=["Date"])[["Date", "Ticker"]]
        df = df.merge(ref, on=["Date", "Ticker"], how="inner")
        print(f"[样本对齐] 使用 {ref_pred_path} 的 OOS 样本")
    else:
        # 还没训练：按与 make_expanding_splits 相同的规则截取 OOS 起点
        first_test_year = df["Date"].dt.year.min() + 5
        df = df[df["Date"].dt.year >= first_test_year].copy()
        print(f"[样本对齐] 未找到 {ref_pred_path}，先按日期 >= {first_test_year} 截取 OOS 区间。"
              f"\n           神经网络训练完后请重跑一次，以便样本完全对齐。")

    out = {}
    for name, rule in [("Long-Only", "long_only"),
                       ("Sgn(Returns)", "sgn_returns"),
                       ("MACD-Baz2015", "macd_baz")]:
        d = df.copy()
        d["signal"] = positions(d, rule)
        d["weight"] = d["signal"] * TARGET_DAILY_VOL / d["vol_daily"]
        d["contrib"] = d["weight"] * d["fwd_ret_1d"]

        daily = build_daily_portfolio(d)
        raw = daily["port_ret"].dropna()
        resc = rescale_to_target_vol(raw).dropna()

        daily["raw_return"] = raw
        daily["rescaled_return"] = resc
        fname = {"Long-Only": "longonly", "Sgn(Returns)": "sgn", "MACD-Baz2015": "macd"}[name]
        daily.to_csv(f"{fname}_oos_daily_returns.csv")

        print(f"\n=== {name} RAW ===")
        for k, v in summary_table(raw).items():
            print(f"{k:<20s} {v:.4f}")

### Entry point

In [5]:
if __name__ == "__main__":
    main()

Loaded 240,991 rows | 51 tickers | 2002-04-08 -> 2026-09-02
[样本对齐] 使用 lstm_oos_predictions.csv 的 OOS 样本

=== Long-Only RAW ===
E[Return]            0.0451
Vol.                 0.0866
Downside Deviation   0.0670
MDD                  0.2347
Sharpe               0.5214
Sortino              0.6740
Calmar               0.1923
% +ve Returns        0.5392
Ave.P/Ave.L          0.9346

=== Sgn(Returns) RAW ===
E[Return]            0.0270
Vol.                 0.0727
Downside Deviation   0.0584
MDD                  0.1703
Sharpe               0.3719
Sortino              0.4630
Calmar               0.1587
% +ve Returns        0.5370
Ave.P/Ave.L          0.9212

=== MACD-Baz2015 RAW ===
E[Return]            0.0081
Vol.                 0.0234
Downside Deviation   0.0180
MDD                  0.0967
Sharpe               0.3455
Sortino              0.4486
Calmar               0.0835
% +ve Returns        0.5260
Ave.P/Ave.L          0.9617
